### Implementação de um MLP do zero usando NumPy

In [1]:
import numpy as np

SIGMOID = 'sigmoid'
RELU = 'relu'
TANH = 'tanh'

class MLP:
    def __init__(self, layers, activation_list, learning_rate=0.01):
        self.layers = layers
        self.activation_list = activation_list
        self.lr = learning_rate
        self.weight_list = []
        self.biase_list = []

        self._init_weights()

    def _init_weights(self):
        for i in range(len(self.layers) - 1):
            w = np.random.randn(self.layers[i], self.layers[i+1]) * np.sqrt(1 / self.layers[i])
            b = np.zeros((1, self.layers[i+1]))
            self.weight_list.append(w)
            self.biase_list.append(b)

    def _activation(self, Z, func):
        if func == SIGMOID:
            return 1 / (1 + np.exp(-Z))
        elif func == RELU:
            return np.maximum(0, Z)
        elif func == TANH:
            return np.tanh(Z)
        else:
            raise ValueError("Activation not allowed")

    def _activation_derivative(self, A, func):
        if func == SIGMOID:
            return A * (1 - A)
        elif func == RELU:
            return (A > 0).astype(float)
        elif func == TANH:
            return 1 - A**2

    def forward(self, X):
        f_pred = X
        f_pred_turn = [f_pred]

        for i in range(len(self.weight_list)):
            Z = f_pred @ self.weight_list[i] + self.biase_list[i]
            f_pred = self._activation(Z, self.activation_list[i])
            f_pred_turn.append(f_pred)

        return f_pred, f_pred_turn

    def compute_loss(self, y_pred, y_true):
        return np.mean((y_pred - y_true) ** 2)

    def backward(self, y_pred_turn, y_true):
        grads_w = []
        grads_b = []

        last_pred = y_pred_turn[-1]
        d_last_pred = (last_pred - y_true)

        for i in reversed(range(len(self.weight_list))):
            A = y_pred_turn[i]
            A_curr = y_pred_turn[i + 1]

            dZ = d_last_pred * self._activation_derivative(A_curr, self.activation_list[i])
            dW = A.T @ dZ / len(y_true)
            dB = np.mean(dZ, axis=0, keepdims=True)

            d_last_pred = dZ @ self.weight_list[i].T

            grads_w.insert(0, dW)
            grads_b.insert(0, dB)

        return grads_w, grads_b

    def update(self, grads_w, grads_b):
        for i in range(len(self.weight_list)):
            self.weight_list[i] -= self.lr * grads_w[i]
            self.biase_list[i] -= self.lr * grads_b[i]

    def fit(self, X, y, epochs=1000):
        for epoch in range(epochs):
            y_pred, y_pred_turn = self.forward(X)
            loss = self.compute_loss(y_pred, y)

            grads_w, grads_b = self.backward(y_pred_turn, y)
            self.update(grads_w, grads_b)

            if epoch % 100 == 0:
                print(f"Epoch {epoch} - Loss: {loss:.4f}")

    def predict(self, X):
        y_pred, _ = self.forward(X)
        return (y_pred > 0.5).astype(int)

### Testando o MLP com um dataset de círculos

In [19]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

X, y = make_circles(n_samples=1000, noise=0.1, factor=0.5)
y = y.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Treinando o MLP próprio e avaliando a acurácia

In [34]:
mlp = MLP(
    layers=[2, 10, 5, 1],
    activation_list=[RELU, RELU, SIGMOID],
    learning_rate=0.01
)

mlp.fit(X_train, y_train, epochs=5000)

preds = mlp.predict(X_test)
accuracy = np.mean(preds == y_test)

print("Accuracy (MLP próprio):", accuracy)

Epoch 0 - Loss: 0.2661
Epoch 100 - Loss: 0.2603
Epoch 200 - Loss: 0.2554
Epoch 300 - Loss: 0.2513
Epoch 400 - Loss: 0.2481
Epoch 500 - Loss: 0.2453
Epoch 600 - Loss: 0.2427
Epoch 700 - Loss: 0.2401
Epoch 800 - Loss: 0.2376
Epoch 900 - Loss: 0.2354
Epoch 1000 - Loss: 0.2334
Epoch 1100 - Loss: 0.2313
Epoch 1200 - Loss: 0.2292
Epoch 1300 - Loss: 0.2271
Epoch 1400 - Loss: 0.2252
Epoch 1500 - Loss: 0.2234
Epoch 1600 - Loss: 0.2217
Epoch 1700 - Loss: 0.2200
Epoch 1800 - Loss: 0.2183
Epoch 1900 - Loss: 0.2167
Epoch 2000 - Loss: 0.2150
Epoch 2100 - Loss: 0.2134
Epoch 2200 - Loss: 0.2118
Epoch 2300 - Loss: 0.2102
Epoch 2400 - Loss: 0.2086
Epoch 2500 - Loss: 0.2071
Epoch 2600 - Loss: 0.2055
Epoch 2700 - Loss: 0.2039
Epoch 2800 - Loss: 0.2024
Epoch 2900 - Loss: 0.2009
Epoch 3000 - Loss: 0.1993
Epoch 3100 - Loss: 0.1978
Epoch 3200 - Loss: 0.1963
Epoch 3300 - Loss: 0.1948
Epoch 3400 - Loss: 0.1933
Epoch 3500 - Loss: 0.1917
Epoch 3600 - Loss: 0.1902
Epoch 3700 - Loss: 0.1887
Epoch 3800 - Loss: 0.187

### Comparando com o MLP do Scikit-learn

In [26]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    hidden_layer_sizes=(10, 5),
    activation='relu',
    learning_rate_init=0.01,
    max_iter=5000,
    random_state=42
)

model.fit(X_train, y_train.ravel())

sk_pred = model.predict(X_test).reshape(-1, 1)
acc_sk = np.mean(sk_pred == y_test)

print("Scikit-learn:", acc_sk)

Scikit-learn: 0.98
